# Laboratory Exercise 3
## Bayesian Networks

## Step 1: Importing Libraries

We use NumPy to generate attribute combinations and Pgmpy to build the Bayesian network and calculate probabilities.

In [ ]:
import numpy as np
from itertools import product
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

/usr/local/lib/python3.13/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


## Step 2: Identifying Conditional Independencies

Given the network structure:

- `O` and `L` are conditionally independent given `S`.
- `O` and `M` are conditionally independent given `S`.

Therefore:

$$
O \perp L \mid S
$$

$$
O \perp M \mid S
$$

The joint probability distribution is:

$$
P(S,O,L,M)=P(S)P(O\mid S)P(L\mid S)P(M\mid S,L)
$$

## Step 3: Building the Bayesian Network

We define the network structure and its conditional probability tables using the values provided in the exercise.

In [ ]:
# Define the network structure
model = DiscreteBayesianNetwork([
    ("S", "O"),
    ("S", "L"),
    ("S", "M"),
    ("L", "M")
])

# Prior probability of spam
cpd_s = TabularCPD(
    variable="S",
    variable_card=2,
    values=[[0.6], [0.4]]
)

# Probability of the word "ofertă"
cpd_o = TabularCPD(
    variable="O",
    variable_card=2,
    values=[
        [0.9, 0.3],
        [0.1, 0.7]
    ],
    evidence=["S"],
    evidence_card=[2]
)

# Probability of containing links
cpd_l = TabularCPD(
    variable="L",
    variable_card=2,
    values=[
        [0.7, 0.2],
        [0.3, 0.8]
    ],
    evidence=["S"],
    evidence_card=[2]
)

# Probability of a long email
cpd_m = TabularCPD(
    variable="M",
    variable_card=2,
    values=[
        [0.8, 0.4, 0.5, 0.1],
        [0.2, 0.6, 0.5, 0.9]
    ],
    evidence=["S", "L"],
    evidence_card=[2, 2]
)

# Add the probability tables to the network
model.add_cpds(cpd_s, cpd_o, cpd_l, cpd_m)

# Check the model
print("Is the model valid?", model.check_model())

Is the model valid? True


## Step 4: Classifying Emails

We calculate the probability that an email is spam given its observed attributes:

$$
P(S=1\mid O,L,M)
$$

If the probability is at least 50%, the email is classified as spam. Otherwise, it is classified as non-spam.

In [ ]:
# Create the inference engine
inference = VariableElimination(model)

# Generate all possible combinations of O, L, and M
combinations = list(product([0, 1], repeat=3))

# Calculate probabilities and classify emails
results = []

for O, L, M in combinations:
    posterior = inference.query(
        variables=["S"],
        evidence={"O": O, "L": L, "M": M},
        show_progress=False
    )

    spam_probability = posterior.values[1]

    classification = (
        "Spam" if spam_probability >= 0.5 else "Non-spam"
    )

    results.append([
        O,
        L,
        M,
        spam_probability,
        classification
    ])

# Display the results
print("O  L  M   P(Spam | O,L,M)   Classification")

for O, L, M, probability, classification in results:
    print(
        f"{O}  {L}  {M}       {probability:.4f}          {classification}"
    )

O  L  M   P(Spam | O,L,M)   Classification
0  0  0       0.0382          Non-spam
0  0  1       0.1370          Non-spam
0  1  0       0.1290          Non-spam
0  1  1       0.4706          Non-spam
1  0  0       0.4545          Non-spam
1  0  1       0.7692          Spam
1  1  0       0.7568          Spam
1  1  1       0.9492          Spam


## Step 5: Interpreting the Results

The network evaluates all eight possible combinations of email attributes.

The results show how the presence of the word "ofertă", links, and a long message affects the probability that an email is spam.

In [ ]:
import pandas as pd

df = pd.DataFrame(
    results,
    columns=[
        "O",
        "L",
        "M",
        "P(Spam | O,L,M)",
        "Classification"
    ]
)

df["P(Spam | O,L,M)"] = (
    df["P(Spam | O,L,M)"] * 100
).round(2)

df

,O,L,M,"P(Spam | O,L,M)",Classification
0,0,0,0,3.82,Non-spam
1,0,0,1,13.70,Non-spam
2,0,1,0,12.90,Non-spam
3,0,1,1,47.06,Non-spam
4,1,0,0,45.45,Non-spam
5,1,0,1,76.92,Spam
6,1,1,0,75.68,Spam
7,1,1,1,94.92,Spam
